# Phase 5 — Model Development & Hyperparameter Tuning

## Uncertainty-Aware Material Selection using Machine Learning

### Objective

Develop a better-generalizing regression model for predicting the
Voigt-Reuss-Hill bulk modulus (`K_VRH`) by controlling model complexity
and performing systematic hyperparameter selection.

### Baseline Motivation

Random Forest achieved the strongest baseline validation performance:

- Validation MAE: 36.60 GPa
- Validation RMSE: 50.32 GPa
- Validation R²: 0.564

However, a substantial train-validation performance gap was observed:

- Training MAE: 12.58 GPa
- Training R²: 0.945

This indicates substantial overfitting in the baseline Random Forest.

### Phase 5 Strategy

1. Reconstruct the same train/validation/test split.
2. Keep the test set completely untouched.
3. Perform cross-validation using the training set only.
4. Tune Random Forest complexity-related hyperparameters.
5. Compare candidate configurations using cross-validated MAE.
6. Evaluate the selected configuration on the validation set.
7. Examine the train-validation generalization gap again.

The test set will remain reserved for final evaluation.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

df = pd.read_csv(
    "../data/processed/clean_materials_data.csv"
)

X = df.drop(columns=["K_VRH"])
y = df["K_VRH"]

print("Dataset:", df.shape)

Dataset: (1181, 5)


In [2]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (944, 4) (944,)
Validation: (118, 4) (118,)
Test: (119, 4) (119,)


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

numerical_features = [
    "nsites",
    "volume",
    "elastic_anisotropy"
]

categorical_features = [
    "space_group"
]

tree_preprocessor = ColumnTransformer([
    (
        "num",
        "passthrough",
        numerical_features
    ),
    (
        "cat",
        OneHotEncoder(
            handle_unknown="ignore"
        ),
        categorical_features
    )
])

rf_baseline = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    )
])

In [4]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    rf_baseline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    },
    n_jobs=-1
)

In [5]:
cv_mae = -cv_results["test_mae"]
cv_rmse = -cv_results["test_rmse"]
cv_r2 = cv_results["test_r2"]

print("Random Forest — 5-Fold Cross-Validation")
print("-----------------------------------------")
print(f"MAE:  {cv_mae.mean():.2f} ± {cv_mae.std():.2f} GPa")
print(f"RMSE: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f} GPa")
print(f"R²:   {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")

Random Forest — 5-Fold Cross-Validation
-----------------------------------------
MAE:  35.42 ± 2.62 GPa
RMSE: 47.29 ± 2.77 GPa
R²:   0.575 ± 0.040


In [6]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "regressor__n_estimators": [
        200, 300, 500, 700
    ],
    
    "regressor__max_depth": [
        None, 5, 8, 12, 16, 20
    ],
    
    "regressor__min_samples_split": [
        2, 5, 10, 15
    ],
    
    "regressor__min_samples_leaf": [
        1, 2, 4, 6, 8
    ],
    
    "regressor__max_features": [
        0.5, 0.75, 1.0, "sqrt"
    ]
}

In [7]:
rf_search = RandomizedSearchCV(
    estimator=rf_baseline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="neg_mean_absolute_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True
)

rf_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'regressor__max_depth': [None, 5, ...], 'regressor__max_features': [0.5, 0.75, ...], 'regressor__min_samples_leaf': [1, 2, ...], 'regressor__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations othe

In [8]:
print("Best parameters:")
print(rf_search.best_params_)

print(
    f"\nBest CV MAE: "
    f"{-rf_search.best_score_:.2f} GPa"
)

Best parameters:
{'regressor__n_estimators': 200, 'regressor__min_samples_split': 2, 'regressor__min_samples_leaf': 1, 'regressor__max_features': 0.75, 'regressor__max_depth': 16}

Best CV MAE: 35.79 GPa


In [9]:
rf_tuned = rf_search.best_estimator_

In [10]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

rf_tuned = rf_search.best_estimator_

# Training predictions
y_train_pred_tuned = rf_tuned.predict(X_train)

# Validation predictions
y_val_pred_tuned = rf_tuned.predict(X_val)


def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return mae, rmse, r2


train_mae, train_rmse, train_r2 = regression_metrics(
    y_train,
    y_train_pred_tuned
)

val_mae, val_rmse, val_r2 = regression_metrics(
    y_val,
    y_val_pred_tuned
)

print("Tuned Random Forest")
print("-------------------")
print(f"Train MAE:       {train_mae:.2f} GPa")
print(f"Validation MAE:  {val_mae:.2f} GPa")
print()
print(f"Train RMSE:      {train_rmse:.2f} GPa")
print(f"Validation RMSE: {val_rmse:.2f} GPa")
print()
print(f"Train R²:        {train_r2:.3f}")
print(f"Validation R²:   {val_r2:.3f}")

Tuned Random Forest
-------------------
Train MAE:       13.90 GPa
Validation MAE:  37.18 GPa

Train RMSE:      18.55 GPa
Validation RMSE: 50.29 GPa

Train R²:        0.935
Validation R²:   0.564


In [11]:
from sklearn.model_selection import cross_validate

leaf_values = [1, 2, 4, 6, 8, 12, 16]

leaf_results = []

for leaf in leaf_values:
    
    model = Pipeline([
        ("preprocessor", tree_preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=leaf,
                random_state=42,
                n_jobs=-1
            )
        )
    ])
    
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        return_train_score=True,
        n_jobs=-1
    )
    
    train_mae_cv = -scores["train_score"].mean()
    val_mae_cv = -scores["test_score"].mean()
    
    leaf_results.append({
        "min_samples_leaf": leaf,
        "CV Train MAE": train_mae_cv,
        "CV Validation MAE": val_mae_cv,
        "Gap": val_mae_cv - train_mae_cv
    })

leaf_results = pd.DataFrame(leaf_results)

leaf_results.round(2)

,min_samples_leaf,CV Train MAE,CV Validation MAE,Gap
0,1,12.99,35.42,22.43
1,2,18.13,36.32,18.19
2,4,25.52,37.47,11.95
3,6,29.12,38.11,8.99
4,8,31.55,39.01,7.47
5,12,35.23,40.99,5.76
6,16,38.02,42.86,4.84


In [12]:
depth_values = [3, 5, 8, 12, 16, 20, None]

depth_results = []

for depth in depth_values:

    model = Pipeline([
        ("preprocessor", tree_preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=depth,
                min_samples_leaf=1,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        return_train_score=True,
        n_jobs=-1
    )

    train_mae_cv = -scores["train_score"].mean()
    val_mae_cv = -scores["test_score"].mean()

    depth_results.append({
        "max_depth": str(depth),
        "CV Train MAE": train_mae_cv,
        "CV Validation MAE": val_mae_cv,
        "Gap": val_mae_cv - train_mae_cv
    })

depth_results = pd.DataFrame(depth_results)

depth_results.round(2)

,max_depth,CV Train MAE,CV Validation MAE,Gap
0,3,47.50,50.29,2.78
1,5,38.40,44.57,6.17
2,8,25.94,37.97,12.03
3,12,16.88,35.54,18.66
4,16,13.82,35.36,21.55
5,20,13.09,35.41,22.32
6,None,12.99,35.42,22.43


In [13]:
feature_values = [0.25, 0.5, 0.75, 1.0, "sqrt"]

feature_results = []

for max_feat in feature_values:

    model = Pipeline([
        ("preprocessor", tree_preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=16,
                min_samples_leaf=1,
                max_features=max_feat,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        return_train_score=True,
        n_jobs=-1
    )

    train_mae_cv = -scores["train_score"].mean()
    val_mae_cv = -scores["test_score"].mean()

    feature_results.append({
        "max_features": str(max_feat),
        "CV Train MAE": train_mae_cv,
        "CV Validation MAE": val_mae_cv,
        "Gap": val_mae_cv - train_mae_cv
    })

feature_results = pd.DataFrame(feature_results)

feature_results.round(2)

,max_features,CV Train MAE,CV Validation MAE,Gap
0,0.25,20.91,39.68,18.77
1,0.5,15.67,36.47,20.80
2,0.75,14.24,35.76,21.52
3,1.0,13.82,35.36,21.55
4,sqrt,30.76,44.10,13.35


In [14]:
rf_candidate = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=300,
            max_depth=16,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=1.0,
            random_state=42,
            n_jobs=-1
        )
    )
])

In [15]:
rf_candidate.fit(X_train, y_train)

y_train_pred_candidate = rf_candidate.predict(X_train)
y_val_pred_candidate = rf_candidate.predict(X_val)

candidate_train_mae = mean_absolute_error(
    y_train,
    y_train_pred_candidate
)

candidate_val_mae = mean_absolute_error(
    y_val,
    y_val_pred_candidate
)

candidate_train_rmse = np.sqrt(
    mean_squared_error(
        y_train,
        y_train_pred_candidate
    )
)

candidate_val_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_pred_candidate
    )
)

candidate_train_r2 = r2_score(
    y_train,
    y_train_pred_candidate
)

candidate_val_r2 = r2_score(
    y_val,
    y_val_pred_candidate
)

print("Phase 5 Random Forest Candidate")
print("--------------------------------")
print(f"Train MAE:       {candidate_train_mae:.2f} GPa")
print(f"Validation MAE:  {candidate_val_mae:.2f} GPa")
print()
print(f"Train RMSE:      {candidate_train_rmse:.2f} GPa")
print(f"Validation RMSE: {candidate_val_rmse:.2f} GPa")
print()
print(f"Train R²:        {candidate_train_r2:.3f}")
print(f"Validation R²:   {candidate_val_r2:.3f}")

Phase 5 Random Forest Candidate
--------------------------------
Train MAE:       13.51 GPa
Validation MAE:  36.51 GPa

Train RMSE:      18.09 GPa
Validation RMSE: 50.22 GPa

Train R²:        0.938
Validation R²:   0.565


## Phase 5 Summary — Model Development

Random Forest was selected for further development because it achieved the
strongest baseline validation performance among the evaluated model families.

### Cross-Validation Baseline

Using 5-fold cross-validation on the training set, the baseline Random Forest
achieved:

- MAE: 35.42 ± 2.62 GPa
- RMSE: 47.29 ± 2.77 GPa
- R²: 0.575 ± 0.040

### Hyperparameter Investigation

A randomized hyperparameter search did not improve cross-validated MAE.
The best randomized-search candidate achieved a CV MAE of 35.79 GPa,
compared with 35.42 GPa for the baseline configuration.

Controlled complexity experiments were therefore performed.

Increasing `min_samples_leaf` reduced the train-validation gap but consistently
increased cross-validation error, indicating increasing underfitting.

Restricting `max_depth` showed that shallow trees substantially underfit the
data. A maximum depth of 16 achieved a CV MAE of 35.36 GPa, which was
essentially comparable to the unrestricted baseline while imposing an explicit
complexity limit.

Reducing `max_features` also degraded cross-validation performance. The best
result was obtained using all available features (`max_features=1.0`).

### Selected Candidate Configuration

The Phase 5 Random Forest candidate uses:

- `n_estimators = 300`
- `max_depth = 16`
- `min_samples_split = 2`
- `min_samples_leaf = 1`
- `max_features = 1.0`
- `random_state = 42`

### Validation Performance

The selected candidate achieved:

- MAE: 36.51 GPa
- RMSE: 50.22 GPa
- R²: 0.565

The improvement over the unrestricted baseline Random Forest was marginal.
Therefore, the main benefit of the selected configuration is not a substantial
increase in predictive accuracy, but an explicit constraint on tree complexity
while preserving essentially the same generalization performance.

A train-validation performance gap remains, indicating residual model variance
and motivating explicit uncertainty estimation in the next stage.

The test set remains untouched and reserved for final evaluation.